In [ ]:
from pyspark.sql.functions import col, sum as spark_sum, avg

In [ ]:
sales_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/mnt/data/cleaned_sales.csv")
products_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/mnt/data/products.csv")

In [ ]:
joined_df = sales_df.join(products_df, on="product_id", how="left")

In [ ]:
category_margins = joined_df.groupBy("category").agg(
    spark_sum("profit").alias("total_profit"),
    spark_sum("revenue").alias("total_revenue"),
    avg("profit_margin").alias("avg_profit_margin")
)
category_margins.show()

In [ ]:
joined_df.write.format("delta").mode("overwrite").saveAsTable("sales_product_joined")

In [ ]:
%sql
SELECT product_id, product_name, SUM(quantity) AS total_units_sold
FROM sales_product_joined
GROUP BY product_id, product_name
ORDER BY total_units_sold DESC
LIMIT 3

In [ ]:
category_margins.write.format("delta").mode("overwrite").save("/mnt/data/category_margins_delta")
category_margins.write.format("csv").option("header", "true").mode("overwrite").save("/mnt/data/category_margins_csv")